In [ ]:
import os
import numpy as np
import scipy.io
import pywt
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import VGG16_Weights
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
def get_label_from_filename(folder, filename):
    label_map = {
        'normal': 0, 'B007': 1, 'B014': 2, 'B021': 3,  
        'IR007': 4, 'IR014': 5, 'IR021': 6, 
        'OR007': 7, 'OR014': 8, 'OR021': 9  
    }
    if 'normal' in folder.lower(): return 0
    prefix = filename.split('_')[0] 
  
    for key in label_map:
        if key in prefix: return label_map[key]
    return -1 

In [ ]:
def extract_1d_signals(root_dir):
    class_signals = {i: [] for i in range(10)}
    print("Aggregating raw 1D files...")
    
    for root, _, files in os.walk(root_dir):
        for file in files:
            if file.endswith('.mat'):
                file_path = os.path.join(root, file)
                label = get_label_from_filename(root, file)
                if label == -1: continue 
                
                mat_data = scipy.io.loadmat(file_path)
                signal_key = [key for key in mat_data.keys() if 'DE_time' in key]
                
                if not signal_key:
                    signal_key = [key for key in mat_data.keys() if type(mat_data[key]) == np.ndarray and len(mat_data[key]) > 1000]
                
                if signal_key:
                    raw_signal = mat_data[signal_key[0]].flatten()
                    class_signals[label].append(raw_signal)
                    
    for label in class_signals:
        if len(class_signals[label]) > 0:
            class_signals[label] = np.concatenate(class_signals[label])
        else:
            print(f"Warning: No data found for class {label}")
            
    return class_signals

In [ ]:
def slice_and_window(class_signals):
    train_samples, val_samples, test_samples = [], [], []
    
    train_imgs, val_imgs, test_imgs = 546, 156, 78
    segment_len = 400
    gap = 2000 
    
    for label, full_1d in class_signals.items():
        train_pts = train_imgs * segment_len
        val_pts = val_imgs * segment_len
        test_pts = test_imgs * segment_len                              
        
        required_length = train_pts + gap + val_pts + gap + test_pts
        if len(full_1d) < required_length:
            print(f"Error: Class {label} only has {len(full_1d)} points, but requires {required_length}.")
            continue
            
        train_1d = full_1d[0 : train_pts]
        val_1d   = full_1d[train_pts + gap : train_pts + gap + val_pts]
        test_1d  = full_1d[train_pts + gap + val_pts + gap : required_length]
        
        for i in range(train_imgs):
            start = i * segment_len
            train_samples.append((train_1d[start : start + segment_len], label))
            
        for i in range(val_imgs):
            start = i * segment_len
            val_samples.append((val_1d[start : start + segment_len], label))
            
        for i in range(test_imgs):
            start = i * segment_len
            test_samples.append((test_1d[start : start + segment_len], label))
            
    return train_samples, val_samples, test_samples

In [ ]:


class CWRUDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)
        
    def generate_cwt_image(self, signal):
        fs = 12000.0  
        fc = 1.0      
        frequencies = np.linspace(6000, 30, 128) 
        scales = (fc * fs) / frequencies
        wavelet = 'cmor5.0-1.0' 
        
        coefficients, _ = pywt.cwt(signal, scales, wavelet, sampling_period=1/fs)
        amplitude = np.abs(coefficients)
        
        amp_min, amp_max = amplitude.min(), amplitude.max()
        if amp_max > amp_min:
            normalized_amp = (amplitude - amp_min) / (amp_max - amp_min)
        else:
            normalized_amp = amplitude
            
        img_8bit = np.uint8(normalized_amp * 255)
        img_color = cv2.applyColorMap(img_8bit, cv2.COLORMAP_JET)
        img_resized = cv2.resize(img_color, (224, 224))
        return cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)

    def __getitem__(self, idx):
        signal_segment, label = self.samples[idx]
        image_np = self.generate_cwt_image(signal_segment)
        
        if self.transform:
            image_tensor = self.transform(image_np)
        else:
            transform_default = transforms.Compose([transforms.ToTensor()])
            image_tensor = transform_default(image_np)
            
        return image_tensor, torch.tensor(label, dtype=torch.long)

cwru_root_path = "/kaggle/input/datasets/onkarraskar/cwru-dataset/CWRU_dataset" 

class_signals = extract_1d_signals(cwru_root_path)
train_samples, val_samples, test_samples = slice_and_window(class_signals)

train_dataset = CWRUDataset(train_samples)
val_dataset = CWRUDataset(val_samples)
test_dataset = CWRUDataset(test_samples)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print("\n" + "="*50)
print("CWRU Dataset Split Summary:")
print("="*50)
print(f"Train : {len(train_dataset)} total images (546 per class)")
print(f"Val   : {len(val_dataset)} total images (156 per class)")
print(f"Test  : {len(test_dataset)} total images (78 per class)")
print("="*50 + "\n")





In [ ]:

def get_pretrained_vgg16(num_classes=10):
    model = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(in_features, num_classes)
    return model

model = get_pretrained_vgg16(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)
num_epochs = 20
best_val_acc = 0.0

In [ ]:

print("Starting Pre-training phase on source dataset...\n")

for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_acc = 100. * correct / total

    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    val_acc = 100. * correct_val / total_val
    print(f"Epoch [{epoch+1:02d}/{num_epochs:02d}] | Train Loss: {running_loss/len(train_loader):.4f} - Train Acc: {train_acc:.2f}% | Val Loss: {val_loss/len(val_loader):.4f} - Val Acc: {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'cwru_pretrained_vgg16.pth')
        print("   --> Saved best model!")

print("\nPre-training complete.")

In [ ]:

print("\nLoading the best saved model for testing...")
model.load_state_dict(torch.load('/kaggle/input/datasets/onkarraskar/cwru-best-weight/cwru_pretrained_vgg16.pth', map_location=device))
model.eval()

test_loss, correct_test, total_test = 0.0, 0, 0
all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

avg_test_loss = test_loss / len(test_loader)
test_acc = 100. * correct_test / total_test

print(f"\n=========================================")
print(f"Final Test Loss: {avg_test_loss:.4f}")
print(f"Final Test Accuracy: {test_acc:.2f}%")
print(f"=========================================\n")

class_names = ['normal', 'B007', 'B014', 'B021', 'IR007', 'IR014', 'IR021', 'OR007', 'OR014', 'OR021']
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (CWRU)')
plt.ylabel('True Labels')
plt.xlabel('Predicted Labels')
plt.xticks(rotation=45)
plt.yticks(rotation=45)
plt.tight_layout()
plt.show()

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))